# Catanatron Tutorial

A hands-on walkthrough of the `catanatron` library: game setup, state representation,
actions, stepping through a game, and observing results.

## 1. Imports

In [ ]:
# Core game objects
from catanatron import Game, Player, Color, GameAccumulator
from catanatron import Action, ActionType, WOOD, BRICK, SHEEP, WHEAT, ORE, RESOURCES

# Built-in bots
from catanatron.models.player import RandomPlayer, SimplePlayer
from catanatron.players.weighted_random import WeightedRandomPlayer
from catanatron.players.search import VictoryPointPlayer

# Board / map
from catanatron.models.map import CatanMap, BASE_MAP_TEMPLATE

# State helpers
from catanatron.state_functions import (
    player_key,
    get_actual_victory_points,
    get_visible_victory_points,
    get_longest_road_color,
    get_largest_army,
    get_longest_road_length,
    get_played_dev_cards,
    get_dev_cards_in_hand,
    get_player_buildings,
    player_num_resource_cards,
)

# Building / card type constants (plain strings, not Python Enums)
from catanatron.models.enums import SETTLEMENT, CITY, ROAD, KNIGHT

print("Imports OK")

## 2. Creating a game

`Game` takes a list of `Player` objects and an optional `seed` for reproducibility.
Player seating order is randomised inside the constructor, so always use `player_key()`
to look up per-player state rather than hard-coding `P0_`, `P1_`, etc.

In [ ]:
players = [
    RandomPlayer(Color.RED),
    RandomPlayer(Color.BLUE),
    RandomPlayer(Color.ORANGE),
    RandomPlayer(Color.WHITE),
]

game = Game(players, seed=42)

print(f"Game ID  : {game.id}")
print(f"Seed     : {game.seed}")
print(f"VP target: {game.vps_to_win}")
print(f"Seating  : {game.state.colors}")

## 3. Board and state representation

All mutable game state lives on `game.state` (a `State` object). The two most important
sub-objects are `state.board` (physical board) and `state.player_state` (flat dict of
per-player scalars keyed by `"P{index}_{FIELD}"`).

In [ ]:
state = game.state
board = state.board

print("=== State summary ===")
print(f"num_turns              : {state.num_turns}")
print(f"current_player_index   : {state.current_player_index}")
print(f"current_color          : {state.current_color()}")
print(f"is_initial_build_phase : {state.is_initial_build_phase}")
print(f"is_discarding          : {state.is_discarding}")
print(f"is_moving_knight       : {state.is_moving_knight}")
print()
print("=== Bank (resource_freqdeck) ===")
# Fixed order: [WOOD, BRICK, SHEEP, WHEAT, ORE]
print(dict(zip([WOOD, BRICK, SHEEP, WHEAT, ORE], state.resource_freqdeck)))
print(f"Dev cards remaining: {len(state.development_listdeck)}")
print()
print("=== Board ===")
print(f"Land tiles : {len(board.map.land_tiles)}")
print(f"Land nodes : {len(board.map.land_nodes)}")
print(f"Robber at  : {board.robber_coordinate}")

### Per-player state

`player_key(state, color)` returns the `"P{i}"` prefix for that color.
Use it to index into `state.player_state` rather than hard-coding indices.

In [ ]:
for color in state.colors:
    key = player_key(state, color)
    ps = state.player_state
    print(
        f"{color.value:8s} ({key})  "
        f"VP={ps[f'{key}_ACTUAL_VICTORY_POINTS']}  "
        f"Roads avail={ps[f'{key}_ROADS_AVAILABLE']}  "
        f"Settlements avail={ps[f'{key}_SETTLEMENTS_AVAILABLE']}  "
        f"Cities avail={ps[f'{key}_CITIES_AVAILABLE']}"
    )

### Map: land tiles and node production

The board uses cube coordinates `(x, y, z)` where `x + y + z == 0`.
`node_production` gives expected resource yield at each node — useful for
evaluating settlement placement quality.

In [ ]:
catan_map = board.map

print("Land tiles (coordinate -> resource, number):")
for coord, tile in sorted(catan_map.land_tiles.items()):
    print(f"  {coord}  {str(tile.resource or 'DESERT'):8s}  roll={tile.number}")

print()
print("Top-5 nodes by total expected resource production:")
node_totals = {
    node_id: sum(counter.values())
    for node_id, counter in catan_map.node_production.items()
}
for node_id, total in sorted(node_totals.items(), key=lambda x: -x[1])[:5]:
    detail = dict(catan_map.node_production[node_id])
    print(f"  node {node_id:2d}  total={total:.3f}  {detail}")

## 4. Actions

`Action` is a namedtuple: `Action(color, action_type, value)`.
The current legal actions are always on `state.playable_actions`.

| `ActionType` | `value` |
|---|---|
| `ROLL` | `None` (intent) / `(int, int)` dice tuple in log |
| `BUILD_SETTLEMENT` | `int` node_id |
| `BUILD_ROAD` | `(int, int)` edge |
| `BUILD_CITY` | `int` node_id |
| `BUY_DEVELOPMENT_CARD` | `None` (intent) / card string in log |
| `PLAY_KNIGHT_CARD` | `None` |
| `PLAY_YEAR_OF_PLENTY` | `(resource, resource)` |
| `PLAY_MONOPOLY` | resource string |
| `MARITIME_TRADE` | `(r, r, r, r, r_wanted)` — 4 offered + 1 wanted, padded with `None` |
| `MOVE_ROBBER` | `(coord, victim_color\|None, stolen_resource\|None)` |
| `DISCARD` | `None` (intent) |
| `END_TURN` | `None` |

In [ ]:
print(f"Current player: {state.current_color()}")
print(f"Legal actions ({len(state.playable_actions)} total):")
for action in state.playable_actions[:10]:
    print(f"  ActionType={action.action_type.name:30s}  value={action.value}")
if len(state.playable_actions) > 10:
    print(f"  ... and {len(state.playable_actions) - 10} more")

In [ ]:
action_types = {a.action_type for a in state.playable_actions}
print("Action types available:", {t.name for t in action_types})

## 5. Stepping through a game

Three ways to advance the game:

| Method | Advances by |
|---|---|
| `game.play()` | Full game to completion |
| `game.play_tick()` | Exactly one player decision |
| `game.execute(action)` | One action you choose manually |

In [ ]:
game2 = Game(
    [RandomPlayer(Color.RED), RandomPlayer(Color.BLUE)],
    seed=7,
)

print("Initial placement phase:")
while game2.state.is_initial_build_phase:
    action = game2.play_tick()
    print(f"  {action.color.value:6s}  {action.action_type.name:25s}  {action.value}")

print(f"\nBuild phase complete. Turns so far: {game2.state.num_turns}")

In [ ]:
game3 = Game(
    [RandomPlayer(Color.RED), RandomPlayer(Color.BLUE)],
    seed=99,
)

print("First 15 actions (manual execute):")
for _ in range(15):
    chosen = game3.state.playable_actions[0]
    logged = game3.execute(chosen)  # mutates state in-place; returns fully-resolved action
    print(f"  {logged.color.value:6s}  {logged.action_type.name:25s}  {logged.value}")

In [ ]:
game4 = Game(
    [RandomPlayer(Color.RED), RandomPlayer(Color.BLUE), RandomPlayer(Color.ORANGE)],
    seed=1,
)
winner = game4.play()
print(f"Winner  : {winner}")
print(f"Turns   : {game4.state.num_turns}")
print(f"Actions : {len(game4.state.actions)}")

In [ ]:
base = Game([RandomPlayer(Color.RED), RandomPlayer(Color.BLUE)], seed=5)
for _ in range(50):
    if base.winning_color() is None:
        base.play_tick()

fork = base.copy()
fork.play()

print(f"Base game still at turn {base.state.num_turns}")
print(f"Fork ended at turn    {fork.state.num_turns}, winner: {fork.winning_color()}")

## 6. Built-in bots

| Class | Behaviour |
|---|---|
| `RandomPlayer` | Uniform random over legal actions |
| `SimplePlayer` | Always picks `playable_actions[0]` |
| `WeightedRandomPlayer` | Random, but weights cities >> settlements >> dev cards |
| `VictoryPointPlayer` | One-ply greedy lookahead maximising actual VP |

Custom bots subclass `Player` and override `decide(self, game, playable_actions) -> Action`.

In [ ]:
race_players = [
    RandomPlayer(Color.RED),
    WeightedRandomPlayer(Color.BLUE),
    VictoryPointPlayer(Color.ORANGE),
]

N = 20
win_counts = {Color.RED: 0, Color.BLUE: 0, Color.ORANGE: 0, None: 0}

for seed in range(N):
    g = Game(race_players, seed=seed)
    winner = g.play()
    win_counts[winner] += 1

print(f"Win rates over {N} games:")
for color, wins in win_counts.items():
    label = color.value if color else "draw"
    print(f"  {label:12s}: {wins:2d}/{N}  ({wins/N*100:.0f}%)")

In [ ]:
import random as _random


class GreedySettlerBot(Player):
    """Prioritise city/settlement builds; fall back to random for everything else."""

    PRIORITY = [
        ActionType.BUILD_CITY,
        ActionType.BUILD_SETTLEMENT,
        ActionType.BUY_DEVELOPMENT_CARD,
        ActionType.BUILD_ROAD,
    ]

    def decide(self, game, playable_actions):
        for target_type in self.PRIORITY:
            candidates = [a for a in playable_actions if a.action_type == target_type]
            if candidates:
                return _random.choice(candidates)
        return _random.choice(playable_actions)


custom_game = Game(
    [GreedySettlerBot(Color.RED), RandomPlayer(Color.BLUE)],
    seed=42,
)
winner = custom_game.play()
print(f"Winner: {winner}  after {custom_game.state.num_turns} turns")

## 7. Observing final game state

In [ ]:
obs_game = Game(
    [WeightedRandomPlayer(Color.RED), WeightedRandomPlayer(Color.BLUE), WeightedRandomPlayer(Color.ORANGE)],
    seed=123,
)
obs_game.play()
s = obs_game.state

print(f"Winner : {obs_game.winning_color()}")
print(f"Turns  : {s.num_turns}")

In [ ]:
header = f"{'Color':10s} {'VP':>4s} {'Wood':>5s} {'Brick':>6s} {'Sheep':>6s} {'Wheat':>6s} {'Ore':>5s} {'HasRoad':>8s} {'HasArmy':>8s}"
print(header)
print("-" * len(header))
for color in s.colors:
    key = player_key(s, color)
    ps = s.player_state
    print(
        f"{color.value:10s} "
        f"{get_actual_victory_points(s, color):4d} "
        f"{ps[f'{key}_WOOD_IN_HAND']:5d} "
        f"{ps[f'{key}_BRICK_IN_HAND']:6d} "
        f"{ps[f'{key}_SHEEP_IN_HAND']:6d} "
        f"{ps[f'{key}_WHEAT_IN_HAND']:6d} "
        f"{ps[f'{key}_ORE_IN_HAND']:5d} "
        f"{str(ps[f'{key}_HAS_ROAD']):>8s} "
        f"{str(ps[f'{key}_HAS_ARMY']):>8s}"
    )

In [ ]:
print("Buildings:")
for color in s.colors:
    settlements = get_player_buildings(s, color, SETTLEMENT)
    cities      = get_player_buildings(s, color, CITY)
    roads       = get_player_buildings(s, color, ROAD)
    print(f"  {color.value:10s}  settlements={settlements}  cities={cities}  roads={len(roads)}")

print()
lr_holder = get_longest_road_color(s)
la_color, la_size = get_largest_army(s)
print(f"Longest Road holder: {lr_holder}")
for color in s.colors:
    print(f"  {color.value}: road length = {get_longest_road_length(s, color)}")
print()
print(f"Largest Army holder: {la_color} (army size {la_size})")
for color in s.colors:
    print(f"  {color.value}: {get_played_dev_cards(s, color, KNIGHT)} knights played")

In [ ]:
print(f"Total actions logged: {len(s.actions)}")
print("Last 10:")
for action in s.actions[-10:]:
    print(f"  {action.color.value:8s}  {action.action_type.name:25s}  {action.value}")

## 8. GameAccumulator — collecting data during a game

`GameAccumulator` is a hook interface: override `before(game)`, `step(game, action)`,
and `after(game)` to collect data alongside a running game without touching game logic.

In [ ]:
from collections import Counter


class ActionTypeCounter(GameAccumulator):
    """Count how many times each ActionType occurs in a single game."""

    def before(self, game):
        self.counts = Counter()

    def step(self, game, action):
        self.counts[action.action_type.name] += 1

    def after(self, game):
        pass


acc = ActionTypeCounter()
acc_game = Game(
    [WeightedRandomPlayer(Color.RED), WeightedRandomPlayer(Color.BLUE)],
    seed=0,
)
acc_game.play(accumulators=[acc])

print("Action type distribution:")
for name, count in sorted(acc.counts.items(), key=lambda x: -x[1]):
    print(f"  {name:30s}: {count}")